In [1]:
from langchain_ollama import OllamaLLM

# MODEL = 'llama3'
MODEL = 'qwen'

model = OllamaLLM(model=MODEL, temperature=4)


In [2]:
# check if model is loaded
model.invoke("Tell me a joke")

'Why did the scarecrow win an award?\n\nBecause he was outstanding in his field! (get it?)'

In [3]:
from langchain_core.output_parsers import StrOutputParser
from langchain.prompts import PromptTemplate

template = """
Answer the question based only on the context below. If you can't 
answer the question, reply "I don't know".

Context: {context}

Question: {question}
"""

prompt = PromptTemplate.from_template(template)

parser = StrOutputParser()

chain = prompt | model | parser 

# chain.invoke("What is quantum physics")

In [4]:
from langchain_chroma import Chroma
from utils import get_embedding_function

CHROMA_PATH = "chroma"
DATA_PATH = "data"

db = Chroma(persist_directory=CHROMA_PATH, embedding_function=get_embedding_function())



In [5]:
def get_context(query_text: str):
    # Search the DB.
    # results = db.similarity_search_with_relevance_scores(query_text, k=5)
    results = db.similarity_search_with_score(query_text, k=5)

    # Store results as context
    context_text = "\n\n---\n\n".join([doc.page_content for doc, _score in results])
    return context_text

In [6]:
def query(query_text: str):
    # Get context from query
    context_text = get_context(query_text)

    print("Context:\n")
    print(context_text)
    print("\n#########")

    response = chain.invoke(
        {
            'context': context_text,
            'question': query_text
        }
    )
    print("\nResponse:")
    print(response)
    

In [7]:
query("Who is the main character?")

Context:

Very soon the Rabbit noticed Alice, as she went hunting about, and called out to her in an angry tone, “Why, Mary Ann, what are you doing out here? Run home this moment, and fetch me a pair of gloves and a fan! Quick, now!” And Alice was so much frightened that she ran off at once in the direction it pointed to, without trying to explain the mistake it had made.

---

to, and, as the doubled-up soldiers were always getting up and walking off to other parts of the ground, Alice soon came to the conclusion that it was a very difficult game indeed.

---

way forwards each time and a long way back, and barking hoarsely all the while, till at last it sat down a good way off, panting, with its tongue hanging out of its mouth, and its great eyes half shut.

---

“Well, perhaps not,” said Alice in a soothing tone: “don’t be angry about it. And yet I wish I could show you our cat Dinah: I think you’d take a fancy to cats if you could only see her. She is such a dear quiet thing,” Alic

In [5]:
from utils import *

rag_model = RAGModel(model='qwen')

rag_model.query("What is the title of the story and the author's name?")

Context:

Alice’s Adventures in Wonderland

by Lewis Carroll

THE MILLENNIUM FULCRUM EDITION 3.0

Contents

CHAPTER I. Down the Rabbit-Hole CHAPTER II. The Pool of Tears CHAPTER III. A Caucus-Race and a Long Tale CHAPTER IV. The Rabbit Sends in a Little Bill CHAPTER V. Advice from a Caterpillar CHAPTER VI. Pig and Pepper CHAPTER VII. A Mad Tea-Party CHAPTER VIII. The Queen’s Croquet-Ground CHAPTER IX. The Mock Turtle’s Story CHAPTER X. The Lobster Quadrille CHAPTER XI. Who Stole the Tarts? CHAPTER XII. Alice’s Evidence

CHAPTER I. Down the Rabbit-Hole

---

“Suppose we change the subject,” the March Hare interrupted, yawning. “I’m getting tired of this. I vote the young lady tells us a story.”

“I’m afraid I don’t know one,” said Alice, rather alarmed at the proposal.

“Then the Dormouse shall!” they both cried. “Wake up, Dormouse!” And they pinched it on both sides at once.

The Dormouse slowly opened his eyes. “I wasn’t asleep,” he said in a hoarse, feeble voice: “I heard every word 

'The title of the story is "Alice\'s Adventures in Wonderland." The author of this story is Lewis Carroll.'

In [8]:
chain.invoke(
    {
        'context': """Content
CHAPTER I. Down the Rabbit-Hole
CHAPTER II. The Pool of Tears
CHAPTER III. A Caucus-Race and a Long Tale
CHAPTER IV. The Rabbit Sends in a Little Bill
CHAPTER V. Advice from a Caterpillar
CHAPTER VI. Pig and Pepper
CHAPTER VII. A Mad Tea-Party
CHAPTER VIII. The Queen’s Croquet-Ground
CHAPTER IX. The Mock Turtle’s Story
CHAPTER X. The Lobster Quadrille
CHAPTER XI. Who Stole the Tarts?
CHAPTER XII. Alice’s Evidence""",
        'question': "How many chapters are there in the content?"
    }
)

'There are 12 chapters in the content.'

In [9]:
teamviewer_password = "ragteamviewer2003k21"